Celda 1: Importación de librerías y carga de datos

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

PATH_INPUT = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\dataset\dataset_limpio.xlsx"
PATH_OUTPUT_DIR = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\resultados"
PATH_GRAFICAS = r"C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\notebooks\prediction\graficas"

os.makedirs(PATH_OUTPUT_DIR, exist_ok=True)
os.makedirs(PATH_GRAFICAS, exist_ok=True)

df = pd.read_excel(PATH_INPUT)

print(f"Dataset cargado: {len(df)} registros")
print(f"Columnas: {list(df.columns)}")

Dataset cargado: 340 registros
Columnas: ['Fecha', 'Forma de pago', 'Ataud_Modelo', 'Ataud_Color', 'Capilla', 'Carroza', 'Carroza flores', 'Cargadores', 'Auto', 'Microbus', 'Monto', 'fecha_sospechosa', 'monto_outlier', 'Monto_winsorizado', 'Ataud_completo', 'Periodo']


Celda 2: Construcción de Transacciones

In [2]:
def mapear_familia_ataud(modelo):
    modelo = modelo.lower().strip()
    if modelo == 'sin_ataud':
        return 'sin_ataud'
    if 'lincoln' in modelo or 'uncol' in modelo or 'mooclo' in modelo:
        return 'Lincoln'
    if 'principe' in modelo or 'princesa' in modelo or 'principit' in modelo or 'monsero' in modelo:
        return 'Principe'
    if 'redondo' in modelo or 'angel' in modelo and 'redondo' in modelo:
        return 'Redondo'
    if 'biblia' in modelo:
        return 'Biblia'
    if 'faraon' in modelo or 'farana' in modelo:
        return 'Faraon'
    if 'copa' in modelo:
        return 'Copa'
    if 'lujial' in modelo or 'lujoso' in modelo or 'lujul' in modelo:
        return 'Lujial'
    if 'americano' in modelo:
        return 'Americano'
    if 'panoramico' in modelo:
        return 'Panoramico'
    if 'llanol' in modelo or 'jencal' in modelo or 'tencal' in modelo:
        return 'Llanol'
    return modelo.title()

df['Ataud_Familia'] = df['Ataud_Modelo'].apply(mapear_familia_ataud)

print(f"Familias de Ataud_Modelo: {df['Ataud_Familia'].nunique()}")
print(df['Ataud_Familia'].value_counts().to_string())

def construir_transaccion(row):
    items = []
    if row['Ataud_Familia'] != 'sin_ataud':
        items.append(f"Ataud_{row['Ataud_Familia']}")
    if row['Ataud_Color'] != 'no_especificado':
        items.append(f"Color_{row['Ataud_Color']}")
    if row['Capilla'] != 'sin_capilla':
        items.append(f"Capilla_{row['Capilla']}")
    if row['Carroza'] == 1:
        items.append('Carroza')
    if row['Carroza flores'] == 1:
        items.append('Carroza_flores')
    if row['Auto'] == 1:
        items.append('Auto')
    if row['Microbus'] == 1:
        items.append('Microbus')
    if row['Cargadores'] > 0:
        items.append(f"Cargadores_{int(row['Cargadores'])}")
    if row['Forma de pago'] != 'no_especificado':
        items.append(f"Pago_{row['Forma de pago']}")
    return items

transacciones = df.apply(construir_transaccion, axis=1).tolist()

todos_items = set()
for t in transacciones:
    todos_items.update(t)

print(f"\nTransacciones construidas: {len(transacciones)}")
print(f"Items unicos: {len(todos_items)}")
print(f"Ejemplo: {transacciones[0]}")

Familias de Ataud_Modelo: 51
Ataud_Familia
Americano             77
Lincoln               50
Imperial              26
sin_ataud             24
Madera                24
Principe              24
Biblia                24
Redondo               18
Modelo                 9
Diamante               6
Parvulo                5
Llanol                 3
Faraon                 3
Desconocido            3
Panuelo                2
Figaro                 2
Lujial                 2
Principal              2
Copa                   2
Panoramico             2
Luis Xv                2
Propio                 1
Antiguo                1
Superior               1
Oval                   1
Apumismo               1
Moderno                1
Laminado               1
Juan Pablo             1
Venta De Ataud         1
Suineral               1
Jumbo                  1
Paramo                 1
Velero                 1
Amaquno                1
Jin Colu De Madera     1
Lineal                 1
Allencada              1
Pinol  

Celda 3: Encoding y Ejecución de Apriori

In [3]:
te = TransactionEncoder()
te_ary = te.fit(transacciones).transform(transacciones)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = apriori(df_encoded, min_support=0.3, use_colnames=True)

print(f"Itemsets frecuentes (soporte >= 0.3): {len(frequent_itemsets)}")
print(f"\nTop 10 itemsets por soporte:")
print(frequent_itemsets.sort_values('support', ascending=False).head(10).to_string(index=False))

Itemsets frecuentes (soporte >= 0.3): 25

Top 10 itemsets por soporte:
 support                                           itemsets
0.847059                               frozenset({Carroza})
0.705882                          frozenset({Cargadores_4})
0.661765                        frozenset({Carroza_flores})
0.652941                 frozenset({Carroza, Cargadores_4})
0.608824               frozenset({Carroza, Carroza_flores})
0.555882          frozenset({Carroza_flores, Cargadores_4})
0.538235                              frozenset({Microbus})
0.529412 frozenset({Carroza, Carroza_flores, Cargadores_4})
0.514706                     frozenset({Microbus, Carroza})
0.476471                frozenset({Microbus, Cargadores_4})


Celda 4: Generación de Reglas de Asociación

In [4]:
def generar_reglas(frequent_itemsets, min_confidence=0.5):
    if len(frequent_itemsets) == 0:
        return pd.DataFrame()
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)
    rules = rules[rules['lift'] >= 1.0]
    rules = rules.sort_values('lift', ascending=False)
    return rules

reglas = generar_reglas(frequent_itemsets, min_confidence=0.5)

print(f"Reglas generadas (confianza >= 0.5, lift >= 1.0): {len(reglas)}")
if len(reglas) > 0:
    print(f"\nTop 5 reglas por lift:")
    print(reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5).to_string(index=False))

Reglas generadas (confianza >= 0.5, lift >= 1.0): 62

Top 5 reglas por lift:
                                       antecedents                                        consequents  support  confidence     lift
frozenset({Carroza_flores, Carroza, Cargadores_4})                              frozenset({Microbus}) 0.394118    0.744444 1.383121
                             frozenset({Microbus}) frozenset({Carroza_flores, Carroza, Cargadores_4}) 0.394118    0.732240 1.383121
                    frozenset({Microbus, Carroza})          frozenset({Carroza_flores, Cargadores_4}) 0.394118    0.765714 1.377475
         frozenset({Carroza_flores, Cargadores_4})                     frozenset({Microbus, Carroza}) 0.394118    0.708995 1.377475
               frozenset({Microbus, Cargadores_4})               frozenset({Carroza_flores, Carroza}) 0.394118    0.827160 1.358621


Celda 5: Interpretación de Reglas

In [5]:
def interpretar_reglas(reglas):
    interpretaciones = []
    for _, row in reglas.iterrows():
        ant = ', '.join(list(row['antecedents']))
        cons = ', '.join(list(row['consequents']))
        conf = row['confidence'] * 100
        lift = row['lift']

        if lift > 1.5:
            sugerencia = "Bundle/descuento conjunto"
        elif lift < 1:
            sugerencia = "Exclusion mutua - no ofrecer juntos"
        else:
            sugerencia = "Comportamiento estandar"

        texto = f"Cuando se contrata [{ant}], tambien se contrata [{cons}] en el {conf:.1f}% de los casos (lift={lift:.2f}). Esto sugiere: {sugerencia}"
        interpretaciones.append(texto)

    return interpretaciones

if len(reglas) > 0:
    interprets = interpretar_reglas(reglas)
    print("Interpretaciones de reglas:\n")
    for i, interp in enumerate(interprets[:10], 1):
        print(f"{i}. {interp}\n")

Interpretaciones de reglas:

1. Cuando se contrata [Carroza_flores, Carroza, Cargadores_4], tambien se contrata [Microbus] en el 74.4% de los casos (lift=1.38). Esto sugiere: Comportamiento estandar

2. Cuando se contrata [Microbus], tambien se contrata [Carroza_flores, Carroza, Cargadores_4] en el 73.2% de los casos (lift=1.38). Esto sugiere: Comportamiento estandar

3. Cuando se contrata [Microbus, Carroza], tambien se contrata [Carroza_flores, Cargadores_4] en el 76.6% de los casos (lift=1.38). Esto sugiere: Comportamiento estandar

4. Cuando se contrata [Carroza_flores, Cargadores_4], tambien se contrata [Microbus, Carroza] en el 70.9% de los casos (lift=1.38). Esto sugiere: Comportamiento estandar

5. Cuando se contrata [Microbus, Cargadores_4], tambien se contrata [Carroza_flores, Carroza] en el 82.7% de los casos (lift=1.36). Esto sugiere: Comportamiento estandar

6. Cuando se contrata [Carroza_flores, Carroza], tambien se contrata [Microbus, Cargadores_4] en el 64.7% de los cas

Celda 6: Visualizaciones

In [6]:
def graficar_soporte_confianza(reglas):
    if len(reglas) == 0:
        return
    plt.figure(figsize=(10, 6))
    plt.scatter(reglas['support'], reglas['confidence'], s=reglas['lift']*50, alpha=0.6, c=reglas['lift'], cmap='YlOrRd')
    plt.colorbar(label='Lift')
    plt.xlabel('Soporte')
    plt.ylabel('Confianza')
    plt.title('Soporte vs Confianza (tamano = Lift)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PATH_GRAFICAS, 'apriori_scatter.png'), dpi=150, bbox_inches='tight')
    plt.close()

def graficar_top_itemsets(frequent_itemsets, top_n=15):
    if len(frequent_itemsets) == 0:
        return
    top = frequent_itemsets.sort_values('support', ascending=False).head(top_n)
    top['itemset'] = top['itemsets'].apply(lambda x: ', '.join(list(x)))
    plt.figure(figsize=(12, 6))
    plt.barh(range(len(top)), top['support'], color='steelblue')
    plt.yticks(range(len(top)), top['itemset'], fontsize=8)
    plt.xlabel('Soporte')
    plt.title(f'Top {top_n} Itemsets Mas Frecuentes')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(PATH_GRAFICAS, 'apriori_top_itemsets.png'), dpi=150, bbox_inches='tight')
    plt.close()

def graficar_heatmap_lift(reglas):
    if len(reglas) == 0:
        return
    ant_labels = [', '.join(list(a)) for a in reglas['antecedents']]
    cons_labels = [', '.join(list(c)) for c in reglas['consequents']]
    lift_matrix = reglas.pivot_table(index='antecedents', columns='consequents', values='lift', aggfunc='max')
    lift_matrix.index = [', '.join(list(x)) for x in lift_matrix.index]
    lift_matrix.columns = [', '.join(list(x)) for x in lift_matrix.columns]

    if lift_matrix.shape[0] > 0 and lift_matrix.shape[1] > 0:
        plt.figure(figsize=(12, 8))
        sns.heatmap(lift_matrix.astype(float), annot=True, fmt='.2f', cmap='YlOrRd')
        plt.title('Lift entre Antecedentes y Consequentes')
        plt.tight_layout()
        plt.savefig(os.path.join(PATH_GRAFICAS, 'apriori_heatmap.png'), dpi=150, bbox_inches='tight')
        plt.close()

graficar_soporte_confianza(reglas)
graficar_top_itemsets(frequent_itemsets)
graficar_heatmap_lift(reglas)

print("Visualizaciones generadas:")
print("  - apriori_scatter.png")
print("  - apriori_top_itemsets.png")
print("  - apriori_heatmap.png")

Visualizaciones generadas:
  - apriori_scatter.png
  - apriori_top_itemsets.png
  - apriori_heatmap.png


Celda 7: Métricas de Validación

In [7]:
def calcular_metricas(df, frequent_itemsets, reglas, transacciones):
    metricas = []
    metricas.append(('total_transacciones', len(df)))
    metricas.append(('total_items_unicos', len(set(item for t in transacciones for item in t))))
    metricas.append(('frequent_itemsets_count', len(frequent_itemsets)))
    metricas.append(('reglas_generadas', len(reglas)))

    if len(reglas) > 0:
        max_lift_idx = reglas['lift'].idxmax()
        metricas.append(('regla_max_lift', str(reglas.loc[max_lift_idx, ['antecedents', 'consequents', 'lift']].to_dict())))

        max_conf_idx = reglas['confidence'].idxmax()
        metricas.append(('regla_max_confianza', str(reglas.loc[max_conf_idx, ['antecedents', 'consequents', 'confidence']].to_dict())))

    if len(frequent_itemsets) > 0:
        max_sup_idx = frequent_itemsets['support'].idxmax()
        metricas.append(('regla_max_soporte', str(frequent_itemsets.loc[max_sup_idx, ['itemsets', 'support']].to_dict())))

    transacciones_set = [set(t) for t in transacciones]
    cubiertas = 0
    for trans in transacciones_set:
        for _, rule in reglas.iterrows():
            if rule['antecedents'].issubset(trans) and rule['consequents'].issubset(trans):
                cubiertas += 1
                break
    coverage = (cubiertas / len(transacciones)) * 100 if len(transacciones) > 0 else 0
    metricas.append(('coverage', f"{coverage:.2f}%"))

    return pd.DataFrame(metricas, columns=['Metrica', 'Valor'])

df_metricas = calcular_metricas(df, frequent_itemsets, reglas, transacciones)
print("Metricas de validacion:")
print(df_metricas.to_string(index=False))

Metricas de validacion:
                Metrica                                                                                                                                         Valor
    total_transacciones                                                                                                                                           340
     total_items_unicos                                                                                                                                           182
frequent_itemsets_count                                                                                                                                            25
       reglas_generadas                                                                                                                                            62
         regla_max_lift {'antecedents': frozenset({'Carroza_flores', 'Carroza', 'Cargadores_4'}), 'consequents': frozenset({'Microbus'}), 'lift': 

Celda 8: Exportación de Resultados

In [8]:
if len(reglas) > 0:
    reglas_export = reglas.copy()
    reglas_export['antecedents'] = reglas_export['antecedents'].apply(lambda x: ', '.join(list(x)))
    reglas_export['consequents'] = reglas_export['consequents'].apply(lambda x: ', '.join(list(x)))
    reglas_export['interpretacion'] = interpretar_reglas(reglas)
    reglas_export.to_excel(os.path.join(PATH_OUTPUT_DIR, 'reglas_asociacion.xlsx'), index=False)

frequent_itemsets_export = frequent_itemsets.copy()
frequent_itemsets_export['itemsets'] = frequent_itemsets_export['itemsets'].apply(lambda x: ', '.join(list(x)))
frequent_itemsets_export.to_excel(os.path.join(PATH_OUTPUT_DIR, 'itemsets_frecuentes.xlsx'), index=False)

df_metricas.to_excel(os.path.join(PATH_OUTPUT_DIR, 'metricas_apriori.xlsx'), index=False)

print(f"Archivos exportados en: {PATH_OUTPUT_DIR}")
print("Pipeline de Apriori completado.")

Archivos exportados en: C:\Users\fabri\OneDrive\Desktop\UPAO\Funeraria_Inventario_Inteligente\data\processed\resultados
Pipeline de Apriori completado.
